# Atera XeniumSlide Data Foundation

This notebook is the lightweight reproducibility record for the completed real-data Atera XeniumSlide foundation sprint. It documents the `pyXenium` canonical object change, the two Atera WTA case builds, the generated artifacts, and the `stGPT validate-data` results used as the data contract for downstream model work.

The notebook is intentionally stored with summary outputs only. It does not embed raw Xenium data, `.zarr` stores, parquet tables, H&E images, contour PNGs, checkpoints, or private generated data.

## Executive Summary

- `pyXenium.io.XeniumSlide` is the canonical real-data object for this pipeline.
- Project terminology is standardized on `XeniumSlide` as the single public object name.
- The raw Atera data under `Y:/long/10X_datasets/Xenium/Atera` was treated as read-only.
- Breast and Cervical Atera WTA cases were standardized into one `XeniumSlide` store per case under `D:/GitHub/stGPT/outputs/xenium_slides/atera`.
- Image context uses contour-segmented H&E crops, not per-cell H&E crops.
- Full transcript tables are not materialized into memory by default; transcript access is represented through streamed point sources where needed.
- `stGPT validate-data` passes for both generated slide configs after the coordinate-space correction.

## Canonical pyXenium Object

`XeniumSlide` is now the canonical dataclass in `pyXenium.io`. Its core fields are:

| Field | Meaning |
|---|---|
| `table` | AnnData table containing the sparse cell x gene expression matrix, `obs`, `var`, `obsm['spatial']`, and Xenium metadata. |
| `points` | Materialized point tables when explicitly requested. |
| `shapes` | Cell boundaries, nucleus boundaries, contour polygons, and related geometry tables. |
| `images` | Registered H&E or morphology image metadata and image pyramids when loaded. |
| `contour_images` | Optional contour-cropped image objects. |
| `metadata` | Case, panel, image-transform, source, and batch/slide metadata. |
| `point_sources` | Streamed sources for point-level data such as transcripts without default full materialization. |

Public I/O APIs added or stabilized for this contract:

```python
from pyXenium.io import (
    XeniumSlide,
    read_xenium_slide,
    write_xenium_slide,
    read_slide,
    write_slide,
)

# New code should use XeniumSlide as the single public object name.
```

`read_xenium(..., as_='slide')` is the preferred path. The documentation and downstream stGPT contracts use `XeniumSlide` only.

## Inputs and Outputs

| Case | Raw Xenium root | Output directory |
|---|---|---|
| Breast | `Y:/long/10X_datasets/Xenium/Atera/WTA_Preview_FFPE_Breast_Cancer_outs` | `D:/GitHub/stGPT/outputs/xenium_slides/atera/breast` |
| Cervical | `Y:/long/10X_datasets/Xenium/Atera/WTA_Preview_FFPE_Cervical_Cancer_outs` | `D:/GitHub/stGPT/outputs/xenium_slides/atera/cervical` |

Expected generated artifacts per case:

- `xenium_slide.zarr`
- `slide_manifest.json`
- `qc_report.json`
- `cell_to_contour.parquet`
- `structure_assignments.csv`
- `contour_patches_manifest.json`
- `contour_patches/*.png`

In [1]:
build_command = (
    "pyxenium slide build-atera "
    "--atera-root Y:/long/10X_datasets/Xenium/Atera "
    "--output-root D:/GitHub/stGPT/outputs/xenium_slides/atera "
    "--extract-contour-images "
    "--max-crop-side-px 1024 "
    "--overwrite"
)
print(build_command)

pyxenium slide build-atera --atera-root Y:/long/10X_datasets/Xenium/Atera --output-root D:/GitHub/stGPT/outputs/xenium_slides/atera --extract-contour-images --max-crop-side-px 1024 --overwrite


## Build Semantics

The builder standardizes each case into a single learning object:

- sparse cell x gene matrix with `obs['cell_id']`, centroid coordinates, and `obsm['spatial']`
- cell and nucleus boundary tables under `shapes`
- Xenium panel metadata in `var` and `uns['xenium_slide']['panel']`
- H&E image path, alignment CSV, affine transform, pixel size, and keypoint residual metadata
- contour polygons from GeoJSON
- cell-to-contour assignment by centroid-in-polygon using spatial indexing
- contour H&E crops capped by `max_crop_side_px=1024`
- batch/slide/patient/organ/stain/scanner metadata when available

The first model-facing image unit is the contour crop. Per-cell H&E crops are deliberately out of scope for this sprint.

In [2]:
validate_commands = [
    "stgpt validate-data --config configs/atera_wta_breast_slide.yaml.example "
    "--output D:/GitHub/stGPT/outputs/xenium_slides/atera/breast/stgpt_qc",
    "stgpt validate-data --config configs/atera_wta_cervical_slide.yaml.example "
    "--output D:/GitHub/stGPT/outputs/xenium_slides/atera/cervical/stgpt_qc",
]
print("\n".join(validate_commands))

stgpt validate-data --config configs/atera_wta_breast_slide.yaml.example --output D:/GitHub/stGPT/outputs/xenium_slides/atera/breast/stgpt_qc
stgpt validate-data --config configs/atera_wta_cervical_slide.yaml.example --output D:/GitHub/stGPT/outputs/xenium_slides/atera/cervical/stgpt_qc


In [3]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

root = Path("D:/GitHub/stGPT/outputs/xenium_slides/atera")
rows = []
for case_dir in [root / "breast", root / "cervical"]:
    manifest = json.loads((case_dir / "slide_manifest.json").read_text(encoding="utf-8"))
    qc = json.loads((case_dir / "qc_report.json").read_text(encoding="utf-8"))
    counts = manifest["counts"]
    rows.append(
        {
            "case_name": manifest["case_name"],
            "organ": manifest["metadata"]["organ"],
            "cells": counts["cells"],
            "genes": counts["genes"],
            "contours": counts["contours"],
            "assigned_cells": counts["assigned_cells"],
            "assignment_coverage": f"{manifest['contours']['cell_assignment_coverage']:.2%}",
            "contour_patches": counts["contour_patches"],
            "qc_status": qc["status"],
            "qc_warnings": ", ".join(qc.get("warnings", [])) or "none",
        }
    )

summary = pd.DataFrame(rows)
Markdown(summary.to_markdown(index=False))

| case_name | organ | cells | genes | contours | assigned_cells | assignment_coverage | contour_patches | qc_status | qc_warnings |
|---|---:|---:|---:|---:|---:|---:|---:|---|---|
| atera_wta_breast | breast | 170,057 | 18,028 | 2,606 | 169,432 | 99.63% | 2,606 | pass | none |
| atera_wta_cervical | cervix | 717,576 | 18,028 | 3,128 | 715,415 | 99.70% | 3,128 | pass | none |


## Coordinate-Space Correction

The first Breast QC pass reported only `50,149 / 170,057 = 29.5%` structure-context coverage. The contour overlay showed nearly full tissue coverage, so this was investigated as an assignment problem rather than an annotation problem.

The cause was a coordinate-space mismatch:

- cell centroids in `cells.parquet` are in Xenium physical units, i.e. microns
- Breast and Cervical contour GeoJSON coordinates are in Xenium pixel coordinates
- the Xenium pixel size is `0.2125 um / pixel`
- assignment must therefore compare contours against `cell_x_um / 0.2125`, `cell_y_um / 0.2125`

`pyXenium` now chooses the assignment coordinate space by comparing cell-coordinate bounds against contour bounds. After the fix, both Atera cases assign more than 99.6% of cells to contours.

In [4]:
print("Breast fixed assignment:   169,432 / 170,057 = 99.63%")
print("Cervical fixed assignment: 715,415 / 717,576 = 99.70%")
print("Coordinate space used for assignments: xenium_pixel")

Breast fixed assignment:   169,432 / 170,057 = 99.63%
Cervical fixed assignment: 715,415 / 717,576 = 99.70%
Coordinate space used for assignments: xenium_pixel


## Artifact Contract

| Artifact | Purpose |
|---|---|
| `xenium_slide.zarr` | Canonical `XeniumSlide` store containing the cell table, geometry, panel metadata, and provenance. |
| `slide_manifest.json` | Case-level manifest with counts, spatial bounds, panel summary, H&E transform metadata, contour source, and artifact paths. |
| `qc_report.json` | pyXenium data-foundation QC status and warnings. |
| `cell_to_contour.parquet` | Cell-to-contour assignment table with original cell coordinates and contour assignment coordinates. |
| `structure_assignments.csv` | Contour-to-structure table derived from contour annotations. |
| `contour_patches_manifest.json` | H&E contour crop manifest with crop path, bbox, pyramid level, transform metadata, and source GeoJSON. |
| `contour_patches/*.png` | Cropped, masked contour H&E regions used as image context. |
| `stgpt_qc/` | stGPT validation outputs: `case_manifest.json`, `qc_report.json`, `qc_report.md`, and `splits.csv`. |

The public repository should track this notebook and documentation only. Generated data artifacts remain under `outputs/` and are intentionally ignored by Git.

## How This Record Should Be Used

Use this notebook as the audit trail before training or evaluating stGPT on the Atera cases. The expected workflow is:

1. Rebuild or verify the `XeniumSlide` outputs with `pyxenium slide build-atera`.
2. Inspect each `slide_manifest.json` and `qc_report.json`.
3. Run `stgpt validate-data` against the generated `xenium_slide.zarr` stores.
4. Train only after QC passes.
5. Evaluate against the QC-generated split files rather than creating ad hoc splits.

No biological claim should be made from this data foundation alone. It establishes the canonical data contract and reproducibility trail for later model, evaluation, and spatho evidence work.